In [49]:
from pathlib import Path
import pandas as pd

RAW_DIR = Path("../data/raw/ligue-1/")
csv_files = sorted(RAW_DIR.glob("*.csv"))


## 2. Data Consolidation: Feature Selection & Concatenation
Extract key match metrics, inject the `Season` identifier, and concatenate all individual seasons into a unified DataFrame.

In [55]:
COLS_TO_KEEP = [
    "Season", "Date", "HomeTeam", "AwayTeam", 
    "FTHG", "FTAG", "FTR", 
    "HS", "AS", "HST", "AST", 
    "HC", "AC", "HY", "AY", "HR", "AR"
]

dfs = []

#Adding SEASON column for each season and concat all the season
for file_path in csv_files:
    df_season = pd.read_csv(file_path)
    df_season["Season"] = file_path.stem[7:11]
    #Keeping only the good collumn 
    df_season = df_season[COLS_TO_KEEP]
    dfs.append(df_season)


df_final = pd.concat(dfs, ignore_index=1)
print(df_final.shape)
print(df_final.isna().sum()) #-> To see if some data are missing in each column 


(2337, 17)
Season      0
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
HS          0
AS          0
HST         0
AST         0
HC          0
AC          0
HY          0
AY          0
HR          0
AR          0
dtype: int64


## 3. Data Cleaning & Schema Normalization
Parse match dates, sort chronologically, standardize column names to snake_case, strip whitespace from team names, and enforce strict integer types.

In [56]:
#Convert string Date as Datetime
df_final["Date"] = pd.to_datetime(df_final["Date"], format="%Y-%m-%d")

# Home_teams == away_teams
home_teams = set(df_final["HomeTeam"].unique())
away_teams = set(df_final["AwayTeam"].unique())
print(len(home_teams) == len(away_teams))

#Check if the name of an team change throught time
print(sorted(list(home_teams)))


#Rename columns to snake_case 
rename_mapping = {
    "Season": "season",
    "Date": "match_date",
    "HomeTeam": "home_team",
    "AwayTeam": "away_team",
    "FTHG": "home_goals",
    "FTAG": "away_goals",
    "FTR": "result",
    "HS": "home_shots",
    "AS": "away_shots",
    "HST": "home_shots_on_target",
    "AST": "away_shots_on_target",
    "HC": "home_corners",
    "AC": "away_corners",
    "HY": "home_yellow_cards",
    "AY": "away_yellow_cards",
    "HR": "home_red_cards",
    "AR": "away_red_cards",
}

df_final = df_final.rename(columns=rename_mapping)

#Strip invisible whitespace from team names
df_final["home_team"] = df_final["home_team"].str.strip()
df_final["away_team"] = df_final["away_team"].str.strip()

int_cols = [
    "home_goals", "away_goals", "home_shots", "away_shots",
    "home_shots_on_target", "away_shots_on_target",
    "home_corners", "away_corners", "home_yellow_cards",
    "away_yellow_cards", "home_red_cards", "away_red_cards"
]

df_final[int_cols] = df_final[int_cols].astype(int)
display(df_final.head(100))


True
['Ajaccio', 'Amiens', 'Angers', 'Auxerre', 'Bordeaux', 'Brest', 'Clermont', 'Dijon', 'Le Havre', 'Lens', 'Lille', 'Lorient', 'Lyon', 'Marseille', 'Metz', 'Monaco', 'Montpellier', 'Nantes', 'Nice', 'Nimes', 'Paris FC', 'Paris SG', 'Reims', 'Rennes', 'St Etienne', 'Strasbourg', 'Toulouse', 'Troyes']


,season,match_date,home_team,away_team,home_goals,away_goals,result,home_shots,away_shots,home_shots_on_target,away_shots_on_target,home_corners,away_corners,home_yellow_cards,away_yellow_cards,home_red_cards,away_red_cards
0,1920,2019-08-09,Monaco,Lyon,0,3,A,7,13,1,3,2,5,2,2,1,0
1,1920,2019-08-10,Marseille,Reims,0,2,A,10,8,2,5,5,3,1,0,0,0
2,1920,2019-08-10,Angers,Bordeaux,3,1,H,14,8,4,3,4,3,2,1,0,0
3,1920,2019-08-10,Brest,Toulouse,1,1,D,16,13,5,3,4,7,0,0,0,0
4,1920,2019-08-10,Dijon,St Etienne,1,2,A,15,12,7,4,12,5,0,2,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1920,2019-10-19,Reims,Montpellier,1,0,H,12,9,5,0,4,8,2,0,0,0
96,1920,2019-10-19,Toulouse,Lille,2,1,H,17,9,6,2,3,4,2,4,0,0
97,1920,2019-10-20,Bordeaux,St Etienne,0,1,A,12,6,6,3,8,5,0,3,0,0
98,1920,2019-10-20,Monaco,Rennes,3,2,H,14,10,5,3,3,3,2,3,0,0


In [57]:
processed_dir = Path("../data/processed")
df_final.to_csv(processed_dir / "ligue1_matches.csv", index=False)
print("Ligue1 saved!")

Ligue1 saved!
